# Assignment 1: Product Co-occurrence Algorithm
**Victoria Essien**

This notebook computes how often pairs of products are bought together in the same basket.
The data is processed in chunks to stay within memory constraints.

#### Generating the test data

In [3]:
# This generates the test data file
# It creates data_1.csv.gz with 65,536 baskets and 256 products

import subprocess
subprocess.run(['python', 'generate_data.py', '--scale', '1'])
print("data file is ready")

Generating dataset with scale: 1
Generated data file.
data file is ready


#### Viewing the raw data

In [5]:
# opening the compressed file to view the basket and product id

import gzip
import os


with gzip.open('data_1.csv.gz', 'rt') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 9:
            break

01b0eb9b-b47d-4647-b576-2c7f05aee0a2,205
01b0eb9b-b47d-4647-b576-2c7f05aee0a2,8
01b0eb9b-b47d-4647-b576-2c7f05aee0a2,109
27183f81-26c7-4b5f-a371-36c039c6af97,55
2d47b4e7-e958-4880-819c-98e5b01ffc25,177
2d47b4e7-e958-4880-819c-98e5b01ffc25,232
2d47b4e7-e958-4880-819c-98e5b01ffc25,180
86775b84-9e6f-42a6-8614-0f586443b858,197
86775b84-9e6f-42a6-8614-0f586443b858,23
86775b84-9e6f-42a6-8614-0f586443b858,70


#### Detecting the number of unique products

In [7]:
# automatically detecting number of unique products in the file
# this means the code works for any scale, not just scale 1

unique_products = set()
with gzip.open('data_1.csv.gz', 'rt') as f:
    for line in f:
        _, product_id = line.strip().split(',')
        unique_products.add(int(product_id))

NUM_PRODUCTS = len(unique_products)
print(f"Detected {NUM_PRODUCTS} unique products in the data")

Detected 256 unique products in the data


#### splitting products into 4 groups so we never load everything into memory at once

In [9]:
# Definitions of sliptting 

NUM_CHUNKS = 4        # split products into 4 groups
NUM_PRODUCTS = 256    # total products in the data
CHUNK_SIZE = NUM_PRODUCTS // NUM_CHUNKS  # products per group = 64

# Print summary
print(f"Total products: {NUM_PRODUCTS}")
print(f"Number of chunks: {NUM_CHUNKS}")
print(f"Products per chunk: {CHUNK_SIZE}")
print()

# Show what each group contains
for chunk in range(NUM_CHUNKS):
    start = chunk * CHUNK_SIZE        # first product in this group
    end = start + CHUNK_SIZE - 1      # last product in this group
    print(f"Chunk {chunk}: products {start} to {end}")

Total products: 256
Number of chunks: 4
Products per chunk: 64

Chunk 0: products 0 to 63
Chunk 1: products 64 to 127
Chunk 2: products 128 to 191
Chunk 3: products 192 to 255


#### Viewing basket and product structure

In [11]:
# reading one line to understand the structure
# left side is basket_id, right side is product_id

with gzip.open('data_1.csv.gz', 'rt') as f:
    
    # Read just the first line
    line = f.readline()
    
    # Show the raw line
    print("Raw line:", line.strip())
    
    # Splitting into two parts at the comma
    parts = line.strip().split(',')
    
    # First part is the basket ID
    basket_id = parts[0]
    
    # Second part is the product ID - convert to integer
    product_id = int(parts[1])
    
    print("Basket ID:", basket_id)
    print("Product ID:", product_id)

Raw line: 01b0eb9b-b47d-4647-b576-2c7f05aee0a2,205
Basket ID: 01b0eb9b-b47d-4647-b576-2c7f05aee0a2
Product ID: 205


#### Grouping products by basket

In [13]:
# reading through the file and grouping products by basket
# showing only the first 5 baskets

current_basket = None
current_products = []
basket_count = 0

with gzip.open('data_1.csv.gz', 'rt') as f:
    for line in f:
        # Splitting line into basket_id and product_id
        basket_id, product_id = line.strip().split(',')
        product_id = int(product_id)
        
        if basket_id != current_basket:
            
            # Printing the previous basket if it exists
            if current_basket is not None:
                print(f"Basket: {current_basket[:8]}... Products: {current_products}")
                basket_count += 1
            
            if basket_count >= 5:
                break
            
            # Start tracking the new basket
            current_basket = basket_id
            current_products = [product_id]
        
        else:
            # Same basket - add product to the list
            current_products.append(product_id)

print(f"\nShowing first 5 baskets")

Basket: 01b0eb9b... Products: [205, 8, 109]
Basket: 27183f81... Products: [55]
Basket: 2d47b4e7... Products: [177, 232, 180]
Basket: 86775b84... Products: [197, 23, 70]
Basket: 52c4a200... Products: [173, 136, 56, 70]

Showing first 5 baskets


#### Generating pairs from one basket

In [15]:
# generating all possible pairs from one example basket
# smaller number always goes first to avoid double counting

products = [144, 66, 107, 92]

pairs = []

# Loop through each product
for i in range(len(products)):
    for j in range(i + 1, len(products)):
        
        # putting smaller number first to avoid double counting
        p1 = min(products[i], products[j])
        p2 = max(products[i], products[j])
        
        pairs.append((p1, p2))

print("Products in basket:", products)
print()
print("All pairs:")
for pair in pairs:
    print(f"  {pair[0]} + {pair[1]}")

print()
print(f"Total pairs: {len(pairs)}")



Products in basket: [144, 66, 107, 92]

All pairs:
  66 + 144
  107 + 144
  92 + 144
  66 + 107
  66 + 92
  92 + 107

Total pairs: 6


#### Counting how many times each pair appears across all baskets

In [17]:
# first attempt - counting all pairs at once
# this works for small data but would crash for large files
# because it loads all pairs into memory at the same time

pair_counts = {}
current_basket = None
current_products = []

with gzip.open('data_1.csv.gz', 'rt') as f:

    for line in f:

        # Split line into basket_id and product_id
        basket_id, product_id = line.strip().split(',')
        product_id = int(product_id)

        if basket_id != current_basket:

            # Generate all pairs from the previous basket
            for i in range(len(current_products)):
                for j in range(i + 1, len(current_products)):

                    #putting smaller number first
                    p1 = min(current_products[i], current_products[j])
                    p2 = max(current_products[i], current_products[j])

                    # Add 1 to the count for this pair
                    # if pair not seen before start at 0 then add 1
                    pair_counts[(p1, p2)] = pair_counts.get((p1, p2), 0) + 1

            # Start tracking the new basket
            current_basket = basket_id
            current_products = [product_id]

        else:
            current_products.append(product_id)

print("Top 10 most common pairs:")
print()

sorted_pairs = sorted(pair_counts.items(), key=lambda x: x[1], reverse=True)
for pair, count in sorted_pairs[:10]:
    print(f"  Product {pair[0]} + Product {pair[1]} → {count} baskets")

print()
print(f"Total unique pairs found: {len(pair_counts)}")

Top 10 most common pairs:

  Product 69 + Product 213 → 22 baskets
  Product 111 + Product 216 → 22 baskets
  Product 0 + Product 43 → 22 baskets
  Product 64 + Product 201 → 21 baskets
  Product 28 + Product 100 → 20 baskets
  Product 30 + Product 251 → 20 baskets
  Product 79 + Product 95 → 20 baskets
  Product 14 + Product 48 → 20 baskets
  Product 19 + Product 220 → 20 baskets
  Product 91 + Product 229 → 20 baskets

Total unique pairs found: 32623


#### First chunking attempt, range based splitting (discovered a problem with the pairs)

In [19]:
# first attempt at chunking - splitting products by range
# problem discovered: pairs where the two products are in different chunks get missed
# for example pair (45, 130) - product 45 is in chunk 0, product 130 is in chunk 2
# this pair never gets counted in either chunk

os.makedirs('results', exist_ok=True)

for chunk in range(NUM_CHUNKS):
    chunk_start = chunk * CHUNK_SIZE
    chunk_end = chunk_start + CHUNK_SIZE
    pair_counts = {}
    current_basket = None
    current_products = []

    with gzip.open('data_1.csv.gz', 'rt') as f:
        for line in f:
            basket_id, product_id = line.strip().split(',')
            product_id = int(product_id)
            if basket_id != current_basket:
                relevant = [p for p in current_products
                           if chunk_start <= p < chunk_end]
                if len(relevant) >= 2:
                    for i in range(len(relevant)):
                        for j in range(i + 1, len(relevant)):
                            p1 = min(relevant[i], relevant[j])
                            p2 = max(relevant[i], relevant[j])
                            pair_counts[(p1, p2)] = pair_counts.get((p1, p2), 0) + 1
                current_basket = basket_id
                current_products = [product_id]
            else:
                current_products.append(product_id)

    print(f"Chunk {chunk} done — products {chunk_start} to {chunk_end-1} — {len(pair_counts)} pairs found")

print()
print("problem: these only add up to ~8000 pairs but we need 32,624 - cross chunk pairs are missing!")



Chunk 0 done — products 0 to 63 — 2015 pairs found
Chunk 1 done — products 64 to 127 — 2016 pairs found
Chunk 2 done — products 128 to 191 — 2016 pairs found
Chunk 3 done — products 192 to 255 — 2016 pairs found

problem: these only add up to ~8000 pairs but we need 32,624 - cross chunk pairs are missing!


#### Fixed approach - hashing pairs to chunks

In [21]:
# fixing the cross chunk problem using hashing
# instead of assigning products to chunks I assign pairs to chunks
# rule: chunk = smaller product id % number of chunks
# this guarantees every pair goes to exactly one chunk — nothing is missed

import csv

for chunk in range(NUM_CHUNKS):
    pair_counts = {}
    current_basket = None
    current_products = []

    with gzip.open('data_1.csv.gz', 'rt') as f:
        for line in f:
            basket_id, product_id = line.strip().split(',')
            product_id = int(product_id)

            if basket_id != current_basket:
                for i in range(len(current_products)):
                    for j in range(i + 1, len(current_products)):
                        p1 = min(current_products[i], current_products[j])
                        p2 = max(current_products[i], current_products[j])
                        # only count this pair if it belongs to the current chunk
                        if p1 % NUM_CHUNKS == chunk:
                            pair_counts[(p1, p2)] = pair_counts.get((p1, p2), 0) + 1
                current_basket = basket_id
                current_products = [product_id]
            else:
                current_products.append(product_id)

    with open(f'results/chunk_{chunk}_results.csv', 'w', newline='') as out:
        writer = csv.writer(out)
        writer.writerow(['product_1', 'product_2', 'baskets'])
        for (p1, p2), count in pair_counts.items():
            writer.writerow([p1, p2, count])

    print(f"Chunk {chunk} done — {len(pair_counts)} pairs found")

print()
print("All chunks processed!")

Chunk 0 done — 8253 pairs found
Chunk 1 done — 8184 pairs found
Chunk 2 done — 8124 pairs found
Chunk 3 done — 8062 pairs found

All chunks processed!


#### Combining all chunk results into one final file

In [23]:
# combining all 4 chunk result files into one final csv

final_counts = {}

for chunk in range(NUM_CHUNKS):
    with open(f'results/chunk_{chunk}_results.csv', 'r') as f:
        reader = csv.reader(f)
        next(reader)  # skip header
        for row in reader:
            p1 = int(row[0])
            p2 = int(row[1])
            count = int(row[2])
            final_counts[(p1, p2)] = final_counts.get((p1, p2), 0) + count
    print(f"Chunk {chunk} results loaded")

with open('final_results.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['product_1', 'product_2', 'baskets'])
    for (p1, p2), count in sorted(final_counts.items()):
        writer.writerow([p1, p2, count])

print()
print(f"Total unique pairs: {len(final_counts)}")
print()
print("Top 10 most common pairs:")
sorted_pairs = sorted(final_counts.items(), key=lambda x: x[1], reverse=True)
for pair, count in sorted_pairs[:10]:
    print(f"  Product {pair[0]} + Product {pair[1]} -> {count} baskets")

Chunk 0 results loaded
Chunk 1 results loaded
Chunk 2 results loaded
Chunk 3 results loaded

Total unique pairs: 32623

Top 10 most common pairs:
  Product 0 + Product 43 -> 22 baskets
  Product 69 + Product 213 -> 22 baskets
  Product 111 + Product 216 -> 22 baskets
  Product 64 + Product 201 -> 21 baskets
  Product 28 + Product 100 -> 20 baskets
  Product 30 + Product 251 -> 20 baskets
  Product 14 + Product 48 -> 20 baskets
  Product 79 + Product 95 -> 20 baskets
  Product 19 + Product 220 -> 20 baskets
  Product 91 + Product 229 -> 20 baskets


#### Summary insights:
1. Read the data and did not load everything all at one
2. Process the data in chunks, one at a time and only keeping pairs that belong to that group of chunk using the hashing rule
3. Save the chunks results as a csv file
4. Combined all 4 chunks to have a final table

#### Final function - runs the full algorithm with a single command

In [26]:
# all steps compiled into one function
# run this cell to get the full result without running each step manually

def run_assignment1(input_file, num_chunks=4):
    """
    Computes product co-occurrences from a basket dataset.
    Processes the data in chunks to stay within memory constraints.
    
    input_file  : path to the gzipped CSV file
    num_chunks  : number of chunks to split the data into
    """

    # Create folders for intermediate and final results
    os.makedirs('chunks', exist_ok=True)
    os.makedirs('results', exist_ok=True)

    print(f"Processing {input_file} in {num_chunks} chunks...")
    print()

    # Step 1 and 2: Process one chunk at a time
    for chunk in range(num_chunks):

        # Empty dictionary for this chunk only
        pair_counts = {}

        current_basket = None
        current_products = []

        # Read through the entire input file
        with gzip.open(input_file, 'rt') as f:

            for line in f:

                # Split line into basket_id and product_id
                basket_id, product_id = line.strip().split(',')
                product_id = int(product_id)

                # When we see a new basket process the previous one
                if basket_id != current_basket:

                    # Generate all pairs from the previous basket
                    for i in range(len(current_products)):
                        for j in range(i + 1, len(current_products)):

                            # Always put smaller number first
                            p1 = min(current_products[i], current_products[j])
                            p2 = max(current_products[i], current_products[j])

                            # Only count pairs that belong to this chunk
                            if p1 % num_chunks == chunk:
                                pair_counts[(p1, p2)] = pair_counts.get((p1, p2), 0) + 1

                    # Start tracking the new basket
                    current_basket = basket_id
                    current_products = [product_id]

                else:
                    # Same basket — add product to the list
                    current_products.append(product_id)

        # Step 3: Save chunk results to a file
        with open(f'results/chunk_{chunk}_results.csv', 'w', newline='') as out:
            writer = csv.writer(out)
            writer.writerow(['product_1', 'product_2', 'baskets'])
            for (p1, p2), count in pair_counts.items():
                writer.writerow([p1, p2, count])

        print(f"  Chunk {chunk} done — {len(pair_counts)} pairs found")

    # Step 4: Combine all chunk results into one final file
    print()
    print("Combining results...")

    final_counts = {}

    for chunk in range(num_chunks):
        with open(f'results/chunk_{chunk}_results.csv', 'r') as f:
            reader = csv.reader(f)
            next(reader)  # skip header
            for row in reader:
                p1, p2, count = int(row[0]), int(row[1]), int(row[2])
                final_counts[(p1, p2)] = final_counts.get((p1, p2), 0) + count

    # Save final results
    with open('final_results.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['product_1', 'product_2', 'baskets'])
        for (p1, p2), count in sorted(final_counts.items()):
            writer.writerow([p1, p2, count])

    print(f"Done! Total unique pairs: {len(final_counts)}")
    print()
    print("Top 10 most common pairs:")
    sorted_pairs = sorted(final_counts.items(), key=lambda x: x[1], reverse=True)
    for pair, count in sorted_pairs[:10]:
        print(f"  Product {pair[0]} + Product {pair[1]} -> {count} baskets")

    return final_counts


# Running the algorithm
results = run_assignment1('data_1.csv.gz', num_chunks=4)

Processing data_1.csv.gz in 4 chunks...

  Chunk 0 done — 8253 pairs found
  Chunk 1 done — 8184 pairs found
  Chunk 2 done — 8124 pairs found
  Chunk 3 done — 8062 pairs found

Combining results...
Done! Total unique pairs: 32623

Top 10 most common pairs:
  Product 0 + Product 43 -> 22 baskets
  Product 69 + Product 213 -> 22 baskets
  Product 111 + Product 216 -> 22 baskets
  Product 64 + Product 201 -> 21 baskets
  Product 28 + Product 100 -> 20 baskets
  Product 30 + Product 251 -> 20 baskets
  Product 14 + Product 48 -> 20 baskets
  Product 79 + Product 95 -> 20 baskets
  Product 19 + Product 220 -> 20 baskets
  Product 91 + Product 229 -> 20 baskets
